In [8]:
import os
import re
import warnings
from datetime import date, datetime

import numpy as np
import openpyxl
import pandas as pd
from sqlalchemy import create_engine, text

# Suppress openpyxl warnings
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")


# DB CONFIG

DB_CONFIG = {
    "drivername": "postgresql+psycopg2",
    "username": "postgres",
    "password": "Volkswagengolf2025",
    "host": "localhost",
    "port": 5432,
    "database": "NML_db",
}

DB_URL = (
    f"{DB_CONFIG['drivername']}://{DB_CONFIG['username']}:{DB_CONFIG['password']}"
    f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
)


# BASIC HELPERS

PROMPT_RX = re.compile(r"^\s*please\s+specify", re.IGNORECASE)

META_COLS = [
    "file_name",
    "certificate_number",
    "calibration_date",
    "instrument_type",
    "manufacturer",
    "serial_number",
    "previous_certificate_number",
    "date_received",
    "calibration_procedure_number",
    "method_used",
    "comments",
    "lab_type",
]

def normalize_column_name(col):
    if pd.isna(col):
        return "unnamed"
    col = str(col).strip().lower()
    col = re.sub(r"[\n\r]+", " ", col)
    col = re.sub(r"\s+", "_", col)
    col = re.sub(r"[^\w\s_]", "", col)
    col = re.sub(r"_+", "_", col)
    return col.strip("_")


def safe_numeric_convert(series):
    if not isinstance(series, pd.Series):
        return series
    series = series.replace(["", "-", "—", "N/A", "n/a", "nan", "NaN", None], np.nan)
    series = series.astype(str).str.replace(",", "", regex=False).str.strip()
    return pd.to_numeric(series, errors="coerce")


def safe_date_convert(value):
    """
    Robust date conversion:
    - datetime/date objects -> date
    - Excel serial numbers -> date
    - strings -> date
    """
    if value is None:
        return None
    if isinstance(value, pd.Timestamp):
        try:
            return value.date()
        except Exception:
            return None
    if isinstance(value, datetime):
        return value.date()
    if isinstance(value, date):
        return value
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        try:
            dt = pd.to_datetime(value, unit="d", origin="1899-12-30", errors="coerce")
            return dt.date() if pd.notna(dt) else None
        except Exception:
            return None
    s = str(value).strip()
    if s in ("", "nan", "NaN", "None"):
        return None
    try:
        dt = pd.to_datetime(s, errors="coerce", dayfirst=True)
        return dt.date() if pd.notna(dt) else None
    except Exception:
        return None


def clean_value(val):
    if val is None:
        return None
    s = str(val).strip()
    if s == "" or s.lower() in {"nan", "none"}:
        return None
    if PROMPT_RX.search(s):
        return None
    return s


def extract_mass_from_string(text):
    if not text or pd.isna(text):
        return None
    text = str(text).lower().strip()
    try:
        value = float(text.replace("kg", "").replace("g", "").strip())
        if value > 1000:
            return value / 1000
        return value
    except Exception:
        pass

    for pattern in [r"(\d+\.?\d*)\s*kg", r"(\d+\.?\d*)\s*g"]:
        m = re.search(pattern, text)
        if m:
            val = float(m.group(1))
            return val if "kg" in pattern else val / 1000
    return None


def detect_lab_type(file_path):
    """Return provisional lab type: 'F1/E2' or 'Large Mass'."""
    try:
        xl = pd.ExcelFile(file_path)
        sheets = [s.lower() for s in xl.sheet_names]
        if any("f1" in s or "e2" in s for s in sheets):
            return "F1/E2"
        if any(("large" in s and "mass" in s) or ("results" in s and "sheet" in s) or "input" in s for s in sheets):
            return "Large Mass"
        return "Unknown"
    except Exception:
        return "Unknown"


# FILE FILTER (SKIP JUNK FILES)

def is_valid_cert_file(filename: str) -> bool:

    name = filename.lower().strip()

    # ignore temporary office files
    if name.startswith("~$"):
        return False

    # ignore obvious junk files
    blocked_words = [
        "draft",
        "copy",
        "template",
        "test",
        "old",
        "backup",
        "notes",
        "sample",
    ]

    if any(word in name for word in blocked_words):
        return False

    return True


# ADMIN LABEL HELPERS

def _cell(df, r, c):
    try:
        v = df.iat[r, c]
        if v is None or (isinstance(v, float) and pd.isna(v)):
            return None
        s = str(v).strip()
        return None if s in ("", "nan", "NaN", "None") else s
    except Exception:
        return None


def _find_label_positions(df, label_regex):
    hits = []
    rx = re.compile(label_regex, re.IGNORECASE)
    for r in range(df.shape[0]):
        for c in range(df.shape[1]):
            v = df.iat[r, c]
            if isinstance(v, str) and rx.search(v.strip()):
                hits.append((r, c))
    return hits


def _value_right(df, r, c, max_look=6):
    for k in range(1, max_look + 1):
        val = _cell(df, r, c + k)
        if val:
            return val
    return None


def _value_below(df, r, c, max_look=6):
    for k in range(1, max_look + 1):
        val = _cell(df, r + k, c)
        if val:
            return val
    return None


def get_labeled_value(df, label_regex, right_look=6, below_look=6):
    hits = _find_label_positions(df, rf"^{label_regex}\s*:?\s*$")
    for (r, c) in hits:
        val = _value_right(df, r, c, right_look) or _value_below(df, r, c, below_look)
        if val:
            return val
    return None


def get_block_below(df, label_regex, below_rows=5):
    hits = _find_label_positions(df, rf"^{label_regex}\s*:?\s*$")
    for (r, c) in hits:
        parts = []
        for k in range(1, below_rows + 1):
            v = _cell(df, r + k, c)
            if v:
                parts.append(v)
        if parts:
            return " ".join(parts)
    return None


# ROBUST DATE EXTRACTION (OPENPYXL, HANDLES MERGED CELLS)

DATE_IN_TEXT_RX = re.compile(
    r"(\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b)|(\b\d{4}[/-]\d{1,2}[/-]\d{1,2}\b)",
    re.IGNORECASE,
)
DATE_KEYWORDS = ["date of calibration", "calibration date", "mean date", "date:"]


def _merged_topleft_value(ws, cell):
    for rng in ws.merged_cells.ranges:
        if cell.coordinate in rng:
            return ws.cell(rng.min_row, rng.min_col).value
    return cell.value


def _try_extract_date_from_string(s: str):
    m = DATE_IN_TEXT_RX.search(s)
    if not m:
        return None
    return safe_date_convert(m.group(0))


def extract_calibration_date_robust(file_path: str):
    try:
        wb = openpyxl.load_workbook(file_path, data_only=True, keep_vba=True)
        admin = None
        for s in wb.sheetnames:
            if s.strip().lower() == "admin":
                admin = s
                break

        sheet_order = [admin] if admin else []
        for s in wb.sheetnames:
            if s not in sheet_order:
                sheet_order.append(s)
        sheet_order = sheet_order[:4]

        for sh in sheet_order:
            ws = wb[sh]
            max_r = min(120, ws.max_row or 120)
            max_c = min(25, ws.max_column or 25)

            for r in range(1, max_r + 1):
                for c in range(1, max_c + 1):
                    v = _merged_topleft_value(ws, ws.cell(r, c))
                    if v is None:
                        continue

                    if isinstance(v, str):
                        low = v.strip().lower()
                        if any(k in low for k in DATE_KEYWORDS):
                            same = _try_extract_date_from_string(v)
                            if same:
                                return same

                            for k in range(1, 7):
                                vv = _merged_topleft_value(ws, ws.cell(r, c + k))
                                dd = safe_date_convert(vv)
                                if dd:
                                    return dd
                                if isinstance(vv, str):
                                    dd2 = _try_extract_date_from_string(vv)
                                    if dd2:
                                        return dd2

                            for k in range(1, 7):
                                vv = _merged_topleft_value(ws, ws.cell(r + k, c))
                                dd = safe_date_convert(vv)
                                if dd:
                                    return dd
                                if isinstance(vv, str):
                                    dd2 = _try_extract_date_from_string(vv)
                                    if dd2:
                                        return dd2
        return None
    except Exception:
        return None

# ============================================================
# REFERENCE WEIGHTS
# ============================================================
ALLOWED_REF_IDS = {"0302", "0312", "0699", "0689"}


def pad4(val):
    if val is None:
        return None
    s = re.sub(r"\D", "", str(val))
    if not s:
        return None
    try:
        return f"{int(s):04d}"
    except Exception:
        return None


def read_reference_id_from_admin(file_path, sheet_name="Admin"):
    """Read Admin!I2; if invalid/missing, scan small region for allowed IDs."""
    try:
        wb = openpyxl.load_workbook(file_path, data_only=True, read_only=True, keep_vba=True)
        if sheet_name not in wb.sheetnames:
            for s in wb.sheetnames:
                if s.strip().lower() == "admin":
                    sheet_name = s
                    break
        if sheet_name not in wb.sheetnames:
            return None

        ws = wb[sheet_name]
        raw = ws["I2"].value
        pid = pad4(raw)
        if pid in ALLOWED_REF_IDS:
            return pid

        for row in ws.iter_rows(min_row=1, max_row=40, min_col=1, max_col=12, values_only=True):
            for cell in row:
                pid = pad4(cell)
                if pid in ALLOWED_REF_IDS:
                    return pid
        return None
    except Exception:
        return None


# HEADER DETECTION (F1/E2 RESULTS)

def find_header_row(df_raw, keywords=None):
    if keywords is None:
        keywords = ["nominal", "conventional", "deviation", "uncertainty", "measurement", "tolerance", "oiml", "serial", "i.d", "value"]

    max_search = min(30, len(df_raw))
    best_row, best_score = None, 0
    for idx in range(max_search):
        row = df_raw.iloc[idx]
        row_text = " ".join([str(val).lower() for val in row.values if pd.notna(val)])
        keyword_count = sum(1 for kw in keywords if kw in row_text)
        non_null_count = row.notna().sum()
        score = keyword_count * 10 + non_null_count
        if score > best_score:
            best_score = score
            best_row = idx
    return best_row if best_row is not None else 0


# TOLERANCE LOOKUP

def get_tolerance_from_table(file_path, nominal_mass_kg, oiml_grade):
    try:
        df_tol = pd.read_excel(file_path, sheet_name="Sheet2", header=None)
        oiml_column_map = {"E1": 11, "E2": 12, "F1": 13, "F2": 14, "M1": 15, "M1-2": 16, "M2": 17, "M3": 18}
        grade_upper = str(oiml_grade).upper().strip() if oiml_grade else None
        if not grade_upper or grade_upper not in oiml_column_map:
            return None

        tol_column = oiml_column_map[grade_upper]
        nominal_col = df_tol.iloc[4:, 10].dropna()
        tolerance_col = df_tol.iloc[4:, tol_column].dropna()

        tol = {}
        for idx in range(min(len(nominal_col), len(tolerance_col))):
            nominal_str = str(nominal_col.iloc[idx]).strip()
            tol_val = tolerance_col.iloc[idx]
            nominal_kg = extract_mass_from_string(nominal_str)
            if nominal_kg and pd.notna(tol_val):
                try:
                    tol[nominal_kg] = float(tol_val)
                except Exception:
                    pass

        if nominal_mass_kg in tol:
            return tol[nominal_mass_kg]

        standard = [200, 100, 50, 20, 10, 5, 2, 1]
        remaining, total = nominal_mass_kg, 0
        for m in standard:
            if m in tol:
                while remaining >= m:
                    total += tol[m]
                    remaining -= m
        if remaining > 0.01:
            return None
        return total

    except Exception:
        return None


# SCHEMA (NO mass_metadata TABLE)

def ensure_schema(engine):
    """
    IMPORTANT:
    - calibration_metadata is UNIVERSAL (NO mass-only columns).
    - reference_weights / number_of_weights / range_of_weights / oiml_grade
      are stored directly in the dedicated results tables.
    - mass_metadata table is NOT created.
    """
    with engine.begin() as conn:
        # Universal metadata
        conn.execute(text("""
        CREATE TABLE IF NOT EXISTS calibration_metadata (
            id SERIAL PRIMARY KEY,
            file_name TEXT,
            certificate_number TEXT,
            calibration_date DATE,
            instrument_type TEXT,
            manufacturer TEXT,
            serial_number TEXT,
            previous_certificate_number TEXT,
            date_received DATE,
            calibration_procedure_number TEXT,
            method_used TEXT,
            comments TEXT,
            lab_type TEXT
        );
        """))

        for col, coltype in [
            ("file_name", "TEXT"),
            ("certificate_number", "TEXT"),
            ("calibration_date", "DATE"),
            ("instrument_type", "TEXT"),
            ("manufacturer", "TEXT"),
            ("serial_number", "TEXT"),
            ("previous_certificate_number", "TEXT"),
            ("date_received", "DATE"),
            ("calibration_procedure_number", "TEXT"),
            ("method_used", "TEXT"),
            ("comments", "TEXT"),
            ("lab_type", "TEXT"),
        ]:
            conn.execute(text(f"ALTER TABLE calibration_metadata ADD COLUMN IF NOT EXISTS {col} {coltype};"))

        exists = conn.execute(text("""
            SELECT 1 FROM pg_constraint
            WHERE conname = 'uq_calibration_metadata_file_name'
        """)).scalar()
        if not exists:
            conn.execute(text("""
                ALTER TABLE calibration_metadata
                ADD CONSTRAINT uq_calibration_metadata_file_name UNIQUE (file_name);
            """))

        # F1/E2 results table
        conn.execute(text("""
        CREATE TABLE IF NOT EXISTS f1_e2_results (
            id SERIAL PRIMARY KEY,
            metadata_id INTEGER REFERENCES calibration_metadata(id) ON DELETE CASCADE,
            certificate_number TEXT,
            previous_certificate_number TEXT,
            weight_id TEXT,
            nominal_value DOUBLE PRECISION,
            conventional_mass DOUBLE PRECISION,
            deviation DOUBLE PRECISION,
            measurement_uncertainty DOUBLE PRECISION,
            oiml_tolerance DOUBLE PRECISION,
            weight_class TEXT,
            reference_weights TEXT,
            number_of_weights TEXT,
            range_of_weights TEXT,
            oiml_grade TEXT
        );
        """))

        # Ensure new columns exist even if table existed from older script
        for col, coltype in [
            ("reference_weights", "TEXT"),
            ("number_of_weights", "TEXT"),
            ("range_of_weights", "TEXT"),
            ("oiml_grade", "TEXT"),
        ]:
            conn.execute(text(f"ALTER TABLE f1_e2_results ADD COLUMN IF NOT EXISTS {col} {coltype};"))

        # Large mass results table
        conn.execute(text("""
        CREATE TABLE IF NOT EXISTS large_mass_results (
            id SERIAL PRIMARY KEY,
            metadata_id INTEGER REFERENCES calibration_metadata(id) ON DELETE CASCADE,
            certificate_number TEXT,
            weight_id TEXT,
            nominal_mass_g DOUBLE PRECISION,
            conventional_mass_g DOUBLE PRECISION,
            deviation_mg DOUBLE PRECISION,
            measurement_uncertainty_mg DOUBLE PRECISION,
            tolerance_mg DOUBLE PRECISION,
            reference_weights TEXT,
            number_of_weights TEXT,
            range_of_weights TEXT,
            oiml_grade TEXT
        );
        """))

        for col, coltype in [
            ("reference_weights", "TEXT"),
            ("number_of_weights", "TEXT"),
            ("range_of_weights", "TEXT"),
            ("oiml_grade", "TEXT"),
        ]:
            conn.execute(text(f"ALTER TABLE large_mass_results ADD COLUMN IF NOT EXISTS {col} {coltype};"))

        # Remove mass_metadata if it exists
        conn.execute(text("DROP TABLE IF EXISTS mass_metadata;"))


def recreate_mass_tables(engine):
    """
    Drops ONLY mass result tables, keeps calibration_metadata.
    Recreates them WITHOUT mass_metadata.
    """
    print("🧹 Recreating ONLY mass result tables (keeps calibration_metadata)...")
    with engine.begin() as conn:
        conn.execute(text("DROP TABLE IF EXISTS large_mass_results CASCADE;"))
        conn.execute(text("DROP TABLE IF EXISTS f1_e2_results CASCADE;"))

        conn.execute(text("""
        CREATE TABLE f1_e2_results (
            id SERIAL PRIMARY KEY,
            metadata_id INTEGER REFERENCES calibration_metadata(id) ON DELETE CASCADE,
            certificate_number TEXT,
            previous_certificate_number TEXT,
            weight_id TEXT,
            nominal_value DOUBLE PRECISION,
            conventional_mass DOUBLE PRECISION,
            deviation DOUBLE PRECISION,
            measurement_uncertainty DOUBLE PRECISION,
            oiml_tolerance DOUBLE PRECISION,
            weight_class TEXT,
            reference_weights TEXT,
            number_of_weights TEXT,
            range_of_weights TEXT,
            oiml_grade TEXT
        );
        """))

        conn.execute(text("""
        CREATE TABLE large_mass_results (
            id SERIAL PRIMARY KEY,
            metadata_id INTEGER REFERENCES calibration_metadata(id) ON DELETE CASCADE,
            certificate_number TEXT,
            weight_id TEXT,
            nominal_mass_g DOUBLE PRECISION,
            conventional_mass_g DOUBLE PRECISION,
            deviation_mg DOUBLE PRECISION,
            measurement_uncertainty_mg DOUBLE PRECISION,
            tolerance_mg DOUBLE PRECISION,
            reference_weights TEXT,
            number_of_weights TEXT,
            range_of_weights TEXT,
            oiml_grade TEXT
        );
        """))

        conn.execute(text("DROP TABLE IF EXISTS mass_metadata;"))

    print("✅ Mass result tables recreated.\n")


# METADATA EXTRACTION

def extract_metadata_mass(file_path, provisional_lab_type):
    """
    Returns (md_common, md_massfields)
      - md_common goes into calibration_metadata
      - md_massfields goes directly into the relevant results table
    """
    xl = pd.ExcelFile(file_path)

    md_common = {
        "file_name": os.path.basename(file_path),
        "certificate_number": None,
        "calibration_date": None,
        "instrument_type": None,
        "manufacturer": None,
        "serial_number": None,
        "previous_certificate_number": None,
        "date_received": None,
        "calibration_procedure_number": None,
        "method_used": None,
        "comments": None,
        "lab_type": None,
    }

    md_massfields = {
        "reference_weights": None,
        "number_of_weights": None,
        "range_of_weights": None,
        "oiml_grade": None,
    }

    if provisional_lab_type == "F1/E2":
        sheet_name = None
        for s in xl.sheet_names:
            if s.strip().lower() == "admin":
                sheet_name = s
                break
        if sheet_name is None:
            sheet_name = xl.sheet_names[0]

        df = pd.read_excel(file_path, sheet_name=sheet_name, header=None, dtype=object)

        md_common["calibration_date"] = (
            extract_calibration_date_robust(file_path)
            or safe_date_convert(get_labeled_value(df, r"Date\s*of\s*Calibration"))
            or safe_date_convert(get_labeled_value(df, r"Calibration\s*Date"))
            or safe_date_convert(get_labeled_value(df, r"Mean\s*Date"))
        )

        md_common["certificate_number"] = clean_value(
            get_labeled_value(df, r"Certificate\s*Number")
            or get_labeled_value(df, r"Cert(\.|ificate)?\s*No")
        )

        md_common["manufacturer"] = clean_value(get_labeled_value(df, r"Manufacturer"))
        model_number = clean_value(get_labeled_value(df, r"Model\s*Number"))
        md_common["instrument_type"] = clean_value(get_labeled_value(df, r"Instrument\s*Type"))
        md_common["serial_number"] = clean_value(get_labeled_value(df, r"Serial\s*Number"))
        md_common["previous_certificate_number"] = clean_value(get_labeled_value(df, r"Previous\s*Certificate\s*Number"))
        md_common["date_received"] = safe_date_convert(get_labeled_value(df, r"Date\s*Received"))
        md_common["calibration_procedure_number"] = clean_value(get_labeled_value(df, r"(NML\s+)?Procedure\s*Number"))

        md_common["method_used"] = clean_value(
            get_labeled_value(df, r"De.?cription\s+of\s+Method\s+Used")
            or get_labeled_value(df, r"Method\s+Used")
        )

        md_common["comments"] = clean_value(
            get_labeled_value(df, r"Comments") or get_block_below(df, r"Comments", below_rows=6)
        )

        md_massfields["reference_weights"] = read_reference_id_from_admin(file_path, sheet_name=sheet_name)
        md_massfields["number_of_weights"] = clean_value(
            get_labeled_value(df, r"Please\s+specify\s+the\s+number\s+of\s+weights")
        )
        md_massfields["range_of_weights"] = clean_value(
            get_labeled_value(df, r"Please\s+specify\s+the\s+range\s+of\s+weights")
        )

        if model_number:
            up = model_number.upper()
            if "E2" in up:
                md_massfields["oiml_grade"] = "E2"
                md_common["lab_type"] = "E2"
            elif "F1" in up:
                md_massfields["oiml_grade"] = "F1"
                md_common["lab_type"] = "F1"
            elif "E1" in up:
                md_massfields["oiml_grade"] = "E1"
                md_common["lab_type"] = "E1"
            else:
                md_common["lab_type"] = "F1"
        else:
            md_common["lab_type"] = "F1"

    else:
        # Large Mass
        sheet_name = None
        for s in xl.sheet_names:
            if s.strip().lower() == "input":
                sheet_name = s
                break
        if sheet_name is None:
            sheet_name = xl.sheet_names[0]

        df = pd.read_excel(file_path, sheet_name=sheet_name, header=None, dtype=object)

        md_common["certificate_number"] = clean_value(get_labeled_value(df, r"Cert\s*No"))
        md_common["previous_certificate_number"] = clean_value(get_labeled_value(df, r"Previous\s*Cert"))
        md_common["calibration_date"] = safe_date_convert(get_labeled_value(df, r"Calibration\s*Date"))
        md_common["instrument_type"] = clean_value(get_labeled_value(df, r"(Unit|Instrument)\s*(Description|Type)"))
        md_common["lab_type"] = "Large Mass"

        # Many large-mass files do not have all of these, so okay if null
        md_massfields["oiml_grade"] = clean_value(get_labeled_value(df, r"OIML\s*Grade"))
        md_massfields["reference_weights"] = clean_value(get_labeled_value(df, r"Reference\s*Weights?"))
        md_massfields["number_of_weights"] = clean_value(get_labeled_value(df, r"Number\s*of\s*Weights?"))
        md_massfields["range_of_weights"] = clean_value(get_labeled_value(df, r"Range\s*of\s*Weights?"))

    for k in list(md_common.keys()):
        if k in ("calibration_date", "date_received"):
            continue
        md_common[k] = clean_value(md_common.get(k))

    for k in list(md_massfields.keys()):
        md_massfields[k] = clean_value(md_massfields.get(k))

    return md_common, md_massfields


# F1/E2 RESULTS EXTRACTION

def extract_f1_e2_measurements(file_path, sheet_name):
    try:
        df_raw = pd.read_excel(file_path, sheet_name=sheet_name, header=None)
        header_row = find_header_row(df_raw)
        print(f"   📍 Header at row {header_row}")

        df = pd.read_excel(file_path, sheet_name=sheet_name, header=header_row)
        original_cols = df.columns.tolist()
        df.columns = [normalize_column_name(col) for col in df.columns]
        col_map = dict(zip(df.columns, original_cols))

        df = df.dropna(how="all", axis=0)

        header_keywords = ["nominal", "conventional", "deviation", "uncertainty", "tolerance"]
        for col in df.columns:
            if df[col].dtype == "object":
                mask = df[col].astype(str).str.lower().apply(lambda x: not any(kw in x for kw in header_keywords))
                df = df[mask]

        df = df.reset_index(drop=True)
        result_df = pd.DataFrame()

        for col in df.columns:
            if col in ["i_d_no", "id_no", "i_d"]:
                result_df["weight_id"] = df[col].astype(str).str.strip()
                print(f"   ✓ weight_id from: {col_map.get(col, col)}")
                break

        if "weight_id" not in result_df.columns:
            print("   ⚠️ No weight_id column found!")
            return pd.DataFrame()

        mask = (
            result_df["weight_id"].notna()
            & (result_df["weight_id"] != "")
            & (result_df["weight_id"] != "nan")
            & (result_df["weight_id"].str.strip() != "")
        )
        result_df = result_df[mask]
        if len(result_df) == 0:
            return pd.DataFrame()
        valid_indices = result_df.index

        for col in df.columns:
            if ("nominal" in col) and (col not in ["nominal1", "nominal_1"]):
                nom = safe_numeric_convert(df.loc[valid_indices, col])
                if nom.notna().sum() > 0:
                    result_df["nominal_value"] = nom
                    print(f"   ✓ nominal_value from: {col_map.get(col, col)}")
                    break

        for col in df.columns:
            if ("conventional" in col) and (col not in ["conventional1", "conventional_1"]):
                result_df["conventional_mass"] = safe_numeric_convert(df.loc[valid_indices, col])
                print(f"   ✓ conventional_mass from: {col_map.get(col, col)}")
                break

        if "nominal_value" not in result_df.columns or result_df["nominal_value"].isna().all():
            if "conventional_mass" in result_df.columns:
                def round_to_nominal(val):
                    if pd.isna(val):
                        return None
                    standard_masses = [20000, 10000, 5000, 2000, 1000, 500, 200, 100, 50, 20, 10, 5, 2, 1]
                    return min(standard_masses, key=lambda x: abs(x - val))
                result_df["nominal_value"] = result_df["conventional_mass"].apply(round_to_nominal)
                print("   ✓ nominal_value derived")

        for col in df.columns:
            if ("deviation" in col) and (col not in ["deviation1", "deviation_1"]):
                result_df["deviation"] = safe_numeric_convert(df.loc[valid_indices, col])
                print(f"   ✓ deviation from: {col_map.get(col, col)}")
                break

        for col in df.columns:
            if (("measurement" in col) or ("uncertainty" in col)) and (col not in ["measurement1", "measurement_1"]):
                result_df["measurement_uncertainty"] = safe_numeric_convert(df.loc[valid_indices, col])
                print(f"   ✓ measurement_uncertainty from: {col_map.get(col, col)}")
                break

        for col in df.columns:
            cl = col.lower()
            if ("oiml" in cl) or ("tolerance" in cl):
                tol_data = df[col].loc[valid_indices]
                tol_data = tol_data.astype(str).str.replace(r"[°&$]", "", regex=True).str.strip()
                tol_numeric = pd.to_numeric(tol_data, errors="coerce")
                if tol_numeric.notna().sum() > 0:
                    result_df["oiml_tolerance"] = tol_numeric
                    print(f"   ✓ oiml_tolerance from: {col_map.get(col, col)}")
                    break

        def is_valid_id(val):
            try:
                float(val)
                if isinstance(val, str) and "." in val and len(val.split(".")[1]) > 2:
                    return False
            except Exception:
                pass
            return True

        result_df = result_df[result_df["weight_id"].apply(is_valid_id)].reset_index(drop=True)
        return result_df

    except Exception as e:
        print(f"   ❌ F1/E2 error: {e}")
        return pd.DataFrame()


# LARGE MASS RESULTS EXTRACTION

def extract_large_mass_measurements(file_path, sheet_name="Input", md_massfields=None):
    try:
        df_input = pd.read_excel(file_path, sheet_name=sheet_name, header=None, dtype=str)
        print("   📊 Reading from Input sheet")

        summary_row = None
        for idx, row in df_input.iterrows():
            if "summary" in str(row[3]).lower():
                summary_row = idx
                break
        if summary_row is None:
            print("   ⚠️ Summary row not found")
            return pd.DataFrame()

        row_data = df_input.iloc[summary_row]

        weight_id_raw = str(row_data[4]).strip() if pd.notna(row_data[4]) else ""
        weight_id = weight_id_raw.replace("/", "").strip()

        nominal_str = str(row_data[5]).strip() if pd.notna(row_data[5]) else ""
        conventional_str = str(row_data[6]).strip() if pd.notna(row_data[6]) else ""
        deviation_str = str(row_data[7]).strip() if pd.notna(row_data[7]) else None

        nominal_mass_kg = extract_mass_from_string(nominal_str)
        try:
            conventional_mass_g = float(conventional_str)
        except Exception:
            return pd.DataFrame()

        if deviation_str and deviation_str not in ["nan", "NaN", ""]:
            try:
                deviation_g = float(deviation_str)
            except Exception:
                deviation_g = conventional_mass_g - (nominal_mass_kg * 1000)
        else:
            deviation_g = conventional_mass_g - (nominal_mass_kg * 1000)

        deviation_mg = deviation_g * 1000

        oiml_grade = None
        for idx, row in df_input.iterrows():
            if "oiml grade" in str(row[0]).lower():
                oiml_grade = str(row[1]).strip()
                break

        if not oiml_grade and md_massfields is not None:
            oiml_grade = md_massfields.get("oiml_grade")

        tolerance_mg = get_tolerance_from_table(file_path, nominal_mass_kg, oiml_grade)
        uncertainty_mg = tolerance_mg / 5 if tolerance_mg else None
        nominal_mass_g = nominal_mass_kg * 1000 if nominal_mass_kg is not None else None

        return pd.DataFrame([{
            "weight_id": weight_id,
            "nominal_mass_g": nominal_mass_g,
            "conventional_mass_g": conventional_mass_g,
            "deviation_mg": deviation_mg,
            "measurement_uncertainty_mg": uncertainty_mg,
            "tolerance_mg": tolerance_mg,
        }])

    except Exception as e:
        print(f"   ❌ Large Mass error: {e}")
        return pd.DataFrame()


# DEDUPE + INSERT

def already_imported(engine, file_name: str) -> bool:
    with engine.connect() as conn:
        return bool(
            conn.execute(
                text("SELECT 1 FROM calibration_metadata WHERE file_name=:fn LIMIT 1"),
                {"fn": file_name},
            ).scalar()
        )


def get_metadata_id(engine, file_name: str):
    with engine.connect() as conn:
        return conn.execute(
            text("SELECT id FROM calibration_metadata WHERE file_name=:fn LIMIT 1"),
            {"fn": file_name},
        ).scalar()


def insert_data(engine, file_path):
    file_name = os.path.basename(file_path)

    if already_imported(engine, file_name):
        print(f"⏭️  Skipping (already imported): {file_name}")
        return True

    print(f"\n{'='*60}")
    print(f"📄 {file_name}")
    print("=" * 60)

    provisional = detect_lab_type(file_path)
    print(f"🔬 Detected: {provisional}")
    if provisional == "Unknown":
        print("⚠️ Unknown lab type")
        return False

    md_common, md_massfields = extract_metadata_mass(file_path, provisional)

    results_sheet = None
    final_lab_type = md_common.get("lab_type") or ("Large Mass" if provisional == "Large Mass" else None)

    try:
        xl = pd.ExcelFile(file_path)
        if provisional == "F1/E2":
            for s in xl.sheet_names:
                if "e2" in s.lower() and "result" in s.lower():
                    results_sheet = s
                    final_lab_type = "E2"
                    break
            if results_sheet is None:
                for s in xl.sheet_names:
                    if "f1" in s.lower() and "result" in s.lower():
                        results_sheet = s
                        final_lab_type = "F1"
                        break
            if results_sheet is None:
                for s in xl.sheet_names:
                    if "result" in s.lower():
                        results_sheet = s
                        final_lab_type = "E2" if "e2" in s.lower() else ("F1" if "f1" in s.lower() else (final_lab_type or "F1"))
                        break
        else:
            final_lab_type = "Large Mass"
    except Exception:
        pass

    md_common["lab_type"] = final_lab_type

    md_row = {k: md_common.get(k) for k in META_COLS}

    try:
        pd.DataFrame([md_row]).to_sql("calibration_metadata", engine, if_exists="append", index=False)
        metadata_id = get_metadata_id(engine, file_name)
        print(f"✅ Metadata inserted (ID: {metadata_id}, lab_type={final_lab_type})")
    except Exception as e:
        msg = str(e).lower()
        if "uq_calibration_metadata_file_name" in msg or "duplicate key value" in msg:
            metadata_id = get_metadata_id(engine, file_name)
            print(f"⏭️  Skipping (duplicate file_name): {file_name}")
            return True
        print(f"❌ Metadata insert error: {e}")
        return False

    try:
        cert_no = md_common.get("certificate_number")

        if final_lab_type in ("F1", "E2"):
            if not results_sheet:
                print("   ⚠️ No results sheet found")
                return False

            print(f"   📊 Sheet: '{results_sheet}'")
            df = extract_f1_e2_measurements(file_path, results_sheet)

            if df.empty:
                print("⚠️ No measurement data")
                return False

            cls = "E2" if "e2" in results_sheet.lower() else ("F1" if "f1" in results_sheet.lower() else None)
            df["weight_class"] = cls
            df["metadata_id"] = metadata_id
            df["certificate_number"] = cert_no
            df["previous_certificate_number"] = md_common.get("previous_certificate_number")

            # store mass-specific fields directly in results table
            df["reference_weights"] = md_massfields.get("reference_weights")
            df["number_of_weights"] = md_massfields.get("number_of_weights")
            df["range_of_weights"] = md_massfields.get("range_of_weights")
            df["oiml_grade"] = md_massfields.get("oiml_grade")

            df.to_sql("f1_e2_results", engine, if_exists="append", index=False)
            print(f"✅ Inserted {len(df)} rows into 'f1_e2_results'")

        elif final_lab_type == "Large Mass":
            df = extract_large_mass_measurements(file_path, sheet_name="Input", md_massfields=md_massfields)
            if df.empty:
                print("⚠️ No measurement data")
                return False

            df["metadata_id"] = metadata_id
            df["certificate_number"] = cert_no
            df["reference_weights"] = md_massfields.get("reference_weights")
            df["number_of_weights"] = md_massfields.get("number_of_weights")
            df["range_of_weights"] = md_massfields.get("range_of_weights")
            df["oiml_grade"] = md_massfields.get("oiml_grade")

            df.to_sql("large_mass_results", engine, if_exists="append", index=False)
            print(f"✅ Inserted {len(df)} rows into 'large_mass_results'")

        else:
            print(f"⚠️ Unhandled lab_type: {final_lab_type}")
            return False

        return True

    except Exception as e:
        print(f"❌ Insert error: {e}")
        return False


# MAIN

def run_import(folder_path=".", recreate_mass=False):
    """
    Put this script in the same folder as incoming Mass .xlsm files.
    - Adds NEW files only (dedupe by file_name).
    - DOES NOT wipe calibration_metadata (so other lab data remains intact).
    - NO mass_metadata table is used.
    - reference_weights / number_of_weights / range_of_weights / oiml_grade
      are stored directly in f1_e2_results / large_mass_results.
    """
    print("🚀 NML Mass Import Tool (universal calibration_metadata + results-only mass fields)")
    print("=" * 88)

    try:
        engine = create_engine(DB_URL)
        with engine.connect() as conn:
            conn.execute(text("SELECT 1"))
        print("✅ Database connected\n")
    except Exception as e:
        print(f"❌ Connection failed: {e}")
        return

    ensure_schema(engine)

    if recreate_mass:
        recreate_mass_tables(engine)

    files = [f for f in os.listdir(folder_path) if f.lower().endswith(".xlsm") and not f.startswith("~$")]
    if not files:
        print(f"⚠️ No .xlsm files in '{folder_path}'")
        return

    print(f"Found {len(files)} file(s)\n")

    ok = 0
    for f in sorted(files):
        if insert_data(engine, os.path.join(folder_path, f)):
            ok += 1

    print(f"\n{'='*88}")
    print("📊 SUMMARY")
    print("=" * 88)
    print(f"Seen in folder: {len(files)}")
    print(f"Handled successfully: {ok}")
    print(f"Errors: {len(files) - ok}")

    try:
        with engine.connect() as conn:
            # show only the tables that should exist
            for table in ["calibration_metadata", "f1_e2_results", "large_mass_results"]:
                cnt = conn.execute(text(f"SELECT COUNT(*) FROM public.{table}")).scalar()
                print(f"   public.{table}: {cnt} records")


    except Exception as e:
        print(f"\n⚠️ Count error: {e}")

    print("\n✅ Mass import complete!")


if __name__ == "__main__":
    # Set recreate_mass=True ONLY if you want to wipe/reload mass RESULTS tables.
    run_import(folder_path=".", recreate_mass=False)

🚀 NML Mass Import Tool (universal calibration_metadata + results-only mass fields)
✅ Database connected

Found 104 file(s)

⏭️  Skipping (already imported): 25-11559-1.xlsm
⏭️  Skipping (already imported): 25-11560-1.xlsm
⏭️  Skipping (already imported): 25-11973.xlsm
⏭️  Skipping (already imported): 25-11974.xlsm
⏭️  Skipping (already imported): 25-13251.xlsm
⏭️  Skipping (already imported): 25-13606.xlsm
⏭️  Skipping (already imported): 25-13902.xlsm
⏭️  Skipping (already imported): 25-13933.xlsm
⏭️  Skipping (already imported): 25-13954.xlsm
⏭️  Skipping (already imported): 25-14372.xlsm
⏭️  Skipping (already imported): 25-14413.xlsm
⏭️  Skipping (already imported): 25-14414.xlsm
⏭️  Skipping (already imported): 25-15166.xlsm
⏭️  Skipping (already imported): 25-15265.xlsm
⏭️  Skipping (already imported): 25-15280.xlsm
⏭️  Skipping (already imported): 25-15281.xlsm
⏭️  Skipping (already imported): 25-15282.xlsm
⏭️  Skipping (already imported): 25-15283.xlsm
⏭️  Skipping (already impo